In [ ]:
import os
import numpy as np

# Data preparation (Task 6)
# NB:
# Total files: 1132 in data/out
# Train: 792, Val: 113, Test: 227

# paths to data
SPLITS_AND_FEATURES_DIR = "data/out/"
LABEL_DIR = "data/cmu_us_slt_arctic/lab/"

# Read train.txt, val.txt, and test.txt to obtain basenames for each split.
def make_basenames(split_file):
    with open(split_file, "r") as file:
        basenames = [line.strip() for line in file]
    return basenames

# Load files as lists
train_ids = make_basenames(os.path.join(SPLITS_AND_FEATURES_DIR, "train.txt"))
test_ids = make_basenames(os.path.join(SPLITS_AND_FEATURES_DIR, "test.txt"))
val_ids = make_basenames(os.path.join(SPLITS_AND_FEATURES_DIR, "val.txt"))

def get_all_arctic_basenames(label_dir):
    basenames = []

    for filename in os.listdir(label_dir):
        if filename.startswith("arctic") and filename.endswith(".lab"):
            basename = filename.replace(".lab", "")
            basenames.append(basename)

    return sorted(basenames)

# phoneme set and a mapping from phoneme symbols to integer indices
def load_label_file(basename):
    label_path = os.path.join(LABEL_DIR, basename + ".lab")
    segments = []
    with open(label_path, "r") as file:
        next(file) # skip the header of the file, its "#"
        for line in file:
            parts = line.strip().split()

            timestamp = float(parts[0])
            phoneme = parts[2]

            segments.append((timestamp, phoneme))

    return segments

def extract_phoneme_set(basenames):
    phonemes = set()

    for basename in basenames:
        segments = load_label_file(basename)
        for _, ph in segments:
            phonemes.add(ph)

    return sorted(list(phonemes))

all_ids = get_all_arctic_basenames(LABEL_DIR)
print("Total files found:", len(all_ids)) #should be 1132!

phoneme_set = extract_phoneme_set(train_ids) # could also be all_ids?
phoneme_to_idx = {p: i for i, p in enumerate(phoneme_set)} 
idx_to_phoneme = {i: p for p, i in phoneme_to_idx.items()}
print("The Phoneme set", phoneme_set) 
print("Phoneme set to ints", phoneme_to_idx) # 40 phonemes mapped to an int (+ pau) ~41 in total
print("Ints to phonemes", idx_to_phoneme)


# Load feature files
def load_feature_matrix(basename):
    """
    (T, D)
    T = number of frames
    D = feature dimension (always 39?)
    """
    feature_path = os.path.join(SPLITS_AND_FEATURES_DIR, basename + ".npy")
    features = np.load(feature_path)
    return features

# Example of feature tuple
features = load_feature_matrix("arctic_a0001")
print("Shape of feature matrix", features.shape)  # -> (334, 39)


# Convert time stamps -> assign each phoneme label a frame
# These timestamps represent when the phoneme becomes active.
# Example: frames between t0 and t1 → phoneme0

def align_frames_to_phonemes(features, segments, phoneme_to_idx, frame_shift=0.01):
    T = features.shape[0]
    labels = np.zeros(T, dtype=int)

    for i in range(len(segments)):

        start_time, phoneme = segments[i]

        if i < len(segments) - 1:
            end_time = segments[i + 1][0]
        else:
            end_time = T * frame_shift

        start_frame = int(start_time / frame_shift)
        end_frame = int(end_time / frame_shift)

        start_frame = max(0, start_frame)
        end_frame = min(T, end_frame)

        labels[start_frame:end_frame] = phoneme_to_idx[phoneme]

    return labels

def process_audio_sample(basename):
    """Combine features and labels"""
    features = load_feature_matrix(basename)
    segments = load_label_file(basename)

    labels = align_frames_to_phonemes(features, segments, phoneme_to_idx)

    assert len(labels) == features.shape[0]

    return features, labels


features, labels = process_audio_sample(train_ids[0])

# equal so data is ready for HMM training! Yibby :D
print("Frames:", features.shape[0])
print("Labels:", len(labels))

# for each audio sample we have
# O1:T = feature vectors (from .npy)
# X1:T = phoneme labels (integers)

def build_dataset(basenames):
    """Apply it to the train.txt set"""
    dataset = []

    for basename in basenames:
        features, labels = process_audio_sample(basename)
        dataset.append((features, labels))

    return dataset

train_data = build_dataset(train_ids)
val_data = build_dataset(val_ids)
test_data = build_dataset(test_ids)



Total files found: 1132
The Phoneme set ['aa', 'ae', 'ah', 'ao', 'aw', 'ax', 'ay', 'b', 'ch', 'd', 'dh', 'eh', 'er', 'ey', 'f', 'g', 'hh', 'ih', 'iy', 'jh', 'k', 'l', 'm', 'n', 'ng', 'ow', 'oy', 'p', 'pau', 'r', 's', 'sh', 't', 'th', 'uh', 'uw', 'v', 'w', 'y', 'z', 'zh']
Phoneme set to ints {'aa': 0, 'ae': 1, 'ah': 2, 'ao': 3, 'aw': 4, 'ax': 5, 'ay': 6, 'b': 7, 'ch': 8, 'd': 9, 'dh': 10, 'eh': 11, 'er': 12, 'ey': 13, 'f': 14, 'g': 15, 'hh': 16, 'ih': 17, 'iy': 18, 'jh': 19, 'k': 20, 'l': 21, 'm': 22, 'n': 23, 'ng': 24, 'ow': 25, 'oy': 26, 'p': 27, 'pau': 28, 'r': 29, 's': 30, 'sh': 31, 't': 32, 'th': 33, 'uh': 34, 'uw': 35, 'v': 36, 'w': 37, 'y': 38, 'z': 39, 'zh': 40}
Ints to phonemes {0: 'aa', 1: 'ae', 2: 'ah', 3: 'ao', 4: 'aw', 5: 'ax', 6: 'ay', 7: 'b', 8: 'ch', 9: 'd', 10: 'dh', 11: 'eh', 12: 'er', 13: 'ey', 14: 'f', 15: 'g', 16: 'hh', 17: 'ih', 18: 'iy', 19: 'jh', 20: 'k', 21: 'l', 22: 'm', 23: 'n', 24: 'ng', 25: 'ow', 26: 'oy', 27: 'p', 28: 'pau', 29: 'r', 30: 's', 31: 'sh', 32: 